In [67]:
from pathlib import Path
import shutil

In [106]:
BASE_DIR = Path('D:/Datasets/face mask')
CLASSES = ['with_mask', 'without_mask', 'mask_weared_incorrect']

In [91]:
from sklearn.model_selection import train_test_split
import xmltodict

annotations_dir = BASE_DIR / 'annotations'
annotations = [annotations_dir for annotations_dir in annotations_dir.iterdir()]
train_annotations, val_annotations = train_test_split(
    [xmltodict.parse(annotation.read_text()) for annotation in annotations], 
    test_size=0.2,
    random_state=42
)

In [140]:
train_annotations[0]['annotation']

{'folder': 'images',
 'filename': 'maksssksksss806.png',
 'size': {'width': '400', 'height': '225', 'depth': '3'},
 'segmented': '0',
 'object': [{'name': 'with_mask',
   'pose': 'Unspecified',
   'truncated': '0',
   'occluded': '0',
   'difficult': '0',
   'bndbox': {'xmin': '29', 'ymin': '34', 'xmax': '62', 'ymax': '68'}},
  {'name': 'with_mask',
   'pose': 'Unspecified',
   'truncated': '0',
   'occluded': '0',
   'difficult': '0',
   'bndbox': {'xmin': '95', 'ymin': '53', 'xmax': '126', 'ymax': '91'}},
  {'name': 'with_mask',
   'pose': 'Unspecified',
   'truncated': '0',
   'occluded': '0',
   'difficult': '0',
   'bndbox': {'xmin': '151', 'ymin': '62', 'xmax': '177', 'ymax': '94'}},
  {'name': 'with_mask',
   'pose': 'Unspecified',
   'truncated': '0',
   'occluded': '0',
   'difficult': '0',
   'bndbox': {'xmin': '188', 'ymin': '65', 'xmax': '221', 'ymax': '103'}},
  {'name': 'with_mask',
   'pose': 'Unspecified',
   'truncated': '0',
   'occluded': '0',
   'difficult': '0',
  

In [149]:
def create_markup_string(obj, size) -> str:
    x_center = ((int(obj['bndbox']['xmin']) + int(obj['bndbox']['xmax'])) / 2) / int(size['width'])
    y_center = ((int(obj['bndbox']['ymin']) + int(obj['bndbox']['ymax'])) / 2) / int(size['height'])
    width = ((int(obj['bndbox']['xmax']) - int(obj['bndbox']['xmin']))) / int(size['width'])
    height = ((int(obj['bndbox']['ymax']) - int(obj['bndbox']['ymin']))) / int(size['height'])
    return f"{CLASSES.index(obj['name'])} {round(x_center,3)} {round(y_center,3)} {round(width,3)} {round(height,3)}\n"

In [ ]:
def prepare_data(annotations, mode):
    data_dir = BASE_DIR / mode
    old_images_dir = BASE_DIR / 'images'
    images_dir = data_dir / 'images'
    labels_dir = data_dir / 'labels'
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    for annotation in annotations:
        #если объктов несколько
        text_label = ''
        if isinstance(annotation['annotation']['object'],list):
            for obj in annotation['annotation']['object']:
                text_label += create_markup_string(obj, annotation['annotation']['size'])
        else:
            text_label += create_markup_string(annotation['annotation']['object'], annotation['annotation']['size'])
        
        #копирование картинки и файла разметки
        shutil.copy(old_images_dir / annotation['annotation']['filename'], 
                    images_dir / annotation['annotation']['filename'])
        
        with (labels_dir / annotation['annotation']['filename'].replace('.png','.txt')).open("w", encoding ="utf-8") as f:
            f.write(text_label)


prepare_data(train_annotations, 'train')
prepare_data(val_annotations, 'val')



In [ ]:
with (BASE_DIR / 'data.yaml').open("w", encoding ="utf-8") as f:
            f.write(f'''
path: {BASE_DIR}
# Имена подпапок
train: train/images
val: val/images

# Число классов
nc: 3

names:
  0: {CLASSES[0]}
  1: {CLASSES[1]}
  2: {CLASSES[2]}          

''')

In [35]:
a['annotation']['object']

[{'name': 'without_mask',
  'pose': 'Unspecified',
  'truncated': '0',
  'occluded': '0',
  'difficult': '0',
  'bndbox': {'xmin': '79', 'ymin': '105', 'xmax': '109', 'ymax': '142'}},
 {'name': 'with_mask',
  'pose': 'Unspecified',
  'truncated': '0',
  'occluded': '0',
  'difficult': '0',
  'bndbox': {'xmin': '185', 'ymin': '100', 'xmax': '226', 'ymax': '144'}},
 {'name': 'without_mask',
  'pose': 'Unspecified',
  'truncated': '0',
  'occluded': '0',
  'difficult': '0',
  'bndbox': {'xmin': '325', 'ymin': '90', 'xmax': '360', 'ymax': '141'}}]